In [ ]:
# [COLAB ONLY] Mount Google Drive and set working directory to Code/
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
project_path = '/content/cervical_image_grading/Code'
    if not os.path.exists(project_path):
        print("Please update project_path to point to your Code folder in Google Drive")
    else:
        os.chdir(project_path)
        print("Working directory set to:", os.getcwd())
        
    print("Installing dependencies...")
    !pip install kagglehub albumentations --quiet
except ImportError:
    print("Not running in Google Colab. Skipping drive mount.")


## 2. Dataset Preparation

In [ ]:
import kagglehub
import shutil
import os

if not os.path.exists('sipakmed_data'):
    print("Downloading dataset using kagglehub...")
    path = kagglehub.dataset_download("prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed")
    print("Downloaded to:", path)
    
    # Copy from cache to our workspace directory
    shutil.copytree(path, 'sipakmed_data')
    print("Dataset ready in sipakmed_data!")
else:
    print("Dataset already exists.")


In [ ]:
import os
import sys

# Navigate into the Code directory where all scripts reside
if not os.getcwd().endswith('Code'):
    if os.path.exists('Code'):
        os.chdir('Code')
        print(f"Working directory changed to: {os.getcwd()}")
    else:
        print("Please ensure the 'Code' directory exists in the same path as this notebook.")


In [ ]:
import os
from PIL import Image

# Setup model_data directory, classes, and anchors
model_data_dir = 'model_data'
os.makedirs(model_data_dir, exist_ok=True)

classes = [
    "Dyskeratotic",
    "Koilocytotic",
    "Metaplastic",
    "Parabasal",
    "Superficial-Intermediate"
]

with open(os.path.join(model_data_dir, 'single_cell.txt'), 'w') as f:
    f.write('\n'.join(classes) + '\n')
with open(os.path.join(model_data_dir, 'lzsp_classes.txt'), 'w') as f:
    f.write('\n'.join(classes) + '\n')

anchors = "12,16, 19,36, 40,28, 36,75, 76,55, 72,146, 142,110, 192,243, 459,401"
with open(os.path.join(model_data_dir, '608_anchors.txt'), 'w') as f:
    f.write(anchors)
with open(os.path.join(model_data_dir, 'yolo_anchors.txt'), 'w') as f:
    f.write(anchors)

print("model_data configuration files created successfully!")

# Scan Kaggle folders and generate lzsp_train20201202.txt
if os.path.exists('sipakmed_data'):
    print("Scanning dataset to generate lzsp_train20201202.txt...")
    lines = []
    categories = ["im_Dyskeratotic", "im_Koilocytotic", "im_Metaplastic", "im_Parabasal", "im_Superficial-Intermediate"]
    for idx, cat in enumerate(categories):
        cat_path = os.path.join('sipakmed_data', cat)
        if os.path.exists(cat_path):
            found = 0
            for root, dirs, files in os.walk(cat_path):
                for file in files:
                    if file.lower().endswith(('.jpg', '.png', '.bmp')):
                        img_path = os.path.join(root, file)
                        try:
                            with Image.open(img_path) as im:
                                w, h = im.size
                            lines.append(f"{os.path.relpath(img_path, '.')} 0,0,{w},{h},{idx}\n")
                            found += 1
                        except:
                            pass
            print(f"Category {cat}: Found {found} images.")
    with open('lzsp_train20201202.txt', 'w') as f:
        f.writelines(lines)
    print(f"Successfully generated lzsp_train20201202.txt with {len(lines)} annotations!")
else:
    print("Error: sipakmed_data folder not found!")


## 3. Data Augmentation

In [ ]:
import cv2
import albumentations as A
from tqdm import tqdm
import numpy as np
import os

AUG_IMG_DIR = "sipakmed_data/aug_images"
os.makedirs(AUG_IMG_DIR, exist_ok=True)

# ShiftScaleRotate is a special case of Affine in newer albumentations
transform = A.Compose([
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.1, 0.1), rotate=(-15, 15), p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.3)
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels']))

annotation_file = 'lzsp_train20201202.txt'

if os.path.exists(annotation_file):
    print("Running offline extensive data augmentation from annotation file...")
    with open(annotation_file, 'r') as f:
        lines = f.readlines()
        
    aug_lines = []
    # Only augment a subset to save time/space, e.g., 2 augmented copies per image
    num_copies = 2
    
    for line in tqdm(lines, desc="Augmenting"):
        parts = line.strip().split()
        if not parts: continue
        img_path = parts[0]
        bboxes_str = parts[1:]
        
        if not os.path.exists(img_path):
            continue
            
        image = cv2.imread(img_path)
        if image is None: continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        bboxes = []
        class_labels = []
        for box_str in bboxes_str:
            b_parts = box_str.split(',')
            if len(b_parts) == 5:
                # pascal_voc format: x_min, y_min, x_max, y_max, class_id
                x_min, y_min, x_max, y_max, cls_id = map(float, b_parts)
                # Ensure valid boxes
                if x_max > x_min and y_max > y_min:
                    bboxes.append([x_min, y_min, x_max, y_max])
                    class_labels.append(int(cls_id))
                    
        if not bboxes:
            continue
            
        for i in range(num_copies):
            try:
                transformed = transform(image=image, bboxes=bboxes, class_labels=class_labels)
                
                # Check if bboxes were retained after transforms (e.g. not cropped out)
                if len(transformed['bboxes']) == 0:
                    continue
                    
                aug_img_name = f"aug_{i}_" + os.path.basename(img_path)
                aug_img_path = os.path.join(AUG_IMG_DIR, aug_img_name)
                
                # Save augmented image
                cv2.imwrite(aug_img_path, cv2.cvtColor(transformed['image'], cv2.COLOR_RGB2BGR))
                
                # Format annotation string
                box_strings = []
                for box, cls in zip(transformed['bboxes'], transformed['class_labels']):
                    # convert float back to int for txt format
                    box_strings.append(f"{int(box[0])},{int(box[1])},{int(box[2])},{int(box[3])},{cls}")
                
                aug_lines.append(f"{os.path.relpath(aug_img_path, '.')} " + " ".join(box_strings) + "\n")
            except Exception as e:
                pass
                
    # Append the augmented annotations to the text file
    if len(aug_lines) > 0:
        with open(annotation_file, 'a') as f:
            f.writelines(aug_lines)
        print(f"Offline augmentation completed! Added {len(aug_lines)} augmented images.")
else:
    print("Annotation file not found, skipping augmentation.")


## 4. Helper Configuration (Base YOLO)

In [ ]:
import os
if not os.path.exists('nets'):
    !git clone https://github.com/bubbliiiing/yolov4-pytorch.git temp_repo
    !mv temp_repo/nets ./nets
    !mv temp_repo/utils ./utils
    !rm -rf temp_repo
    print("Standard base helper packages set up!")


In [ ]:
# [CRITICAL COLAB FIX]
# The original bubbliiiing repo has a broken YOLOLoss call and hardcodes weight paths.
# This cell automatically patches train.py so it won't crash during training.
import os
if os.path.exists('train.py'):
    with open('train.py', 'r', encoding='utf-8') as f:
        code = f.read()
    code = code.replace('model_path = "model_data/yolo4_weights.pth"', 'model_path = ""')
    code = code.replace('smoooth_label, Cuda', 'Cuda')
    with open('train.py', 'w', encoding='utf-8') as f:
        f.write(code)
    print("train.py patched for Colab successfully!")
else:
    print("train.py not found. Please ensure the repository was cloned.")
